In [1]:
%pip install xgboost lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: C:\Users\PRIYANKA\Desktop\EVRsystem\.venv\Scripts\python.exe -m pip install --upgrade pip


# Import Libraries 


In [16]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler, StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score, f1_score, accuracy_score, precision_score, recall_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier
import warnings
warnings.filterwarnings('ignore')

# Step 1 Load Splits
def load_splits(data_dir='../data/'):
    train = pd.read_parquet(os.path.join(data_dir, 'train.parquet'))
    val = pd.read_parquet(os.path.join(data_dir, 'val.parquet'))
    test = pd.read_parquet(os.path.join(data_dir, 'test.parquet'))
    return train, val, test

train_df, val_df, test_df = load_splits()
print(f"Splits loaded. Train: {train_df.shape}, Val: {val_df.shape}, Test: {test_df.shape}")

Splits loaded. Train: (922424, 25), Val: (197663, 25), Test: (197663, 25)


### Step 2 Target - Binary Classification for Model A

In [17]:
# Model A Target: high_utilization
threshold = 0.7
for df in [train_df, val_df, test_df]:
    df['high_utilization'] = (df['utilization_rate'] >= threshold).astype(int)

print("Target 'high_utilization' created (Threshold=0.7). Class Distribution (Train):")
print(train_df['high_utilization'].value_counts(normalize=True))

Target 'high_utilization' created (Threshold=0.7). Class Distribution (Train):
high_utilization
0    0.79468
1    0.20532
Name: proportion, dtype: float64


### Step 3 Modular Evaluation Functions

In [18]:
def print_regression_metrics(model, name, X_train, y_train, X_val, y_val, X_test, y_test,
                              y_pred_train_override=None, y_pred_val_override=None, y_pred_test_override=None):
    print(f"\n--- {name} Performance Metrics ---")
    summary = {}
    sets = [
        ('Train', X_train, y_train, y_pred_train_override),
        ('Val', X_val, y_val, y_pred_val_override),
        ('Test', X_test, y_test, y_pred_test_override)
    ]
    
    for s_name, X, y, override in sets:
        preds = model.predict(X)
        if override is not None:
            preds = override
            
        rmse = np.sqrt(mean_squared_error(y, preds))
        r2 = r2_score(y, preds)
        summary[s_name] = {'RMSE': rmse, 'R2': r2}
    
    results_df = pd.DataFrame(summary).T
    print(results_df)

def print_classification_metrics(model, name, X_train, y_train, X_val, y_val, X_test, y_test):
    print(f"\n--- {name} Performance Metrics ---")
    summary = {}
    sets = [('Train', X_train, y_train), ('Val', X_val, y_val), ('Test', X_test, y_test)]
    
    for s_name, X, y in sets:
        preds = model.predict(X)
        f1 = f1_score(y, preds, average='weighted')
        acc = accuracy_score(y, preds)
        precision = precision_score(y, preds, average='weighted')
        recall = recall_score(y, preds, average='weighted')
        summary[s_name] = {'F1_Weighted': f1, 'Accuracy': acc, 'Precision_Weighted': precision, 'Recall_Weighted': recall}
    
    results_df = pd.DataFrame(summary).T
    print(results_df)

### Model A High Utilization (high_utilization)

In [19]:
# Features 
num_features_a = ['hour_of_day', 'day_of_week', 'traffic_congestion_index', 'is_peak_hour', 'month']
cat_features_a = ['location_type']
features_a = num_features_a + cat_features_a

# Data splits
X_train_a, y_train_a = train_df[features_a], train_df['high_utilization']
X_val_a,   y_val_a   = val_df[features_a],   val_df['high_utilization']
X_test_a,  y_test_a  = test_df[features_a],  test_df['high_utilization']

# Pipeline with Categorical Handling (location_type needs encoding, unlike the earlier all-numeric feature set)
preprocessor_a = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), num_features_a),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_a)
])

model_a = Pipeline([
    ('preprocessor', preprocessor_a),
    ('classifier', LGBMClassifier(n_estimators=300, max_depth=4, learning_rate=0.1, random_state=42))
])

# Train
model_a.fit(X_train_a, y_train_a)

# Evaluate
print_classification_metrics(model_a, 'Model A Final', X_train_a, y_train_a, X_val_a, y_val_a, X_test_a, y_test_a)


[LightGBM] [Info] Number of positive: 189392, number of negative: 733032
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007393 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 57
[LightGBM] [Info] Number of data points in the train set: 922424, number of used features: 6
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.205320 -> initscore=-1.353370
[LightGBM] [Info] Start training from score -1.353370
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gai

#### Model A metrics with threshold


In [20]:
# Threshold-tuned evaluation for Model A (addresses low recall on high_utilization)
from sklearn.metrics import classification_report

THRESHOLD_A = 0.30

def evaluate_with_threshold(model, name, X, y, threshold):
    probs = model.predict_proba(X)[:, 1]
    preds = (probs >= threshold).astype(int)
    print(f"\n--- {name} (threshold={threshold}) ---")
    print(classification_report(y, preds, target_names=['low_utilization(0)', 'high_utilization(1)']))

evaluate_with_threshold(model_a, 'Model A Train', X_train_a, y_train_a, THRESHOLD_A)
evaluate_with_threshold(model_a, 'Model A Val',   X_val_a,   y_val_a,   THRESHOLD_A)
evaluate_with_threshold(model_a, 'Model A Test',  X_test_a,  y_test_a,  THRESHOLD_A)


--- Model A Train (threshold=0.3) ---
                     precision    recall  f1-score   support

 low_utilization(0)       0.95      0.84      0.89    733032
high_utilization(1)       0.58      0.84      0.69    189392

           accuracy                           0.84    922424
          macro avg       0.77      0.84      0.79    922424
       weighted avg       0.88      0.84      0.85    922424


--- Model A Val (threshold=0.3) ---
                     precision    recall  f1-score   support

 low_utilization(0)       0.95      0.84      0.90    157394
high_utilization(1)       0.58      0.84      0.69     40269

           accuracy                           0.84    197663
          macro avg       0.77      0.84      0.79    197663
       weighted avg       0.88      0.84      0.85    197663


--- Model A Test (threshold=0.3) ---
                     precision    recall  f1-score   support

 low_utilization(0)       0.95      0.84      0.89    156900
high_utilization(1)      

### Model B Session Duration (avg_session_duration_mins)

In [21]:
# Features
cat_features_b = ['charger_type']
num_features_b = ['power_output_kw']
features_b = num_features_b + cat_features_b

# Data splits
X_train_b, y_train_b = train_df[features_b], train_df['avg_session_duration_mins']
X_val_b,   y_val_b   = val_df[features_b],   val_df['avg_session_duration_mins']
X_test_b,  y_test_b  = test_df[features_b],  test_df['avg_session_duration_mins']

# Pipeline with Random Forest Regressor
preprocessor_b = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())]), num_features_b),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_b)
])

model_b = Pipeline([
    ('preprocessor', preprocessor_b),
    ('regressor', RandomForestRegressor(n_estimators=100, max_depth=6, max_features='sqrt', random_state=42))
])

# Train
model_b.fit(X_train_b, y_train_b)

# Evaluate
print_regression_metrics(model_b, 'Model B', X_train_b, y_train_b, X_val_b, y_val_b, X_test_b, y_test_b)


--- Model B Performance Metrics ---
            RMSE        R2
Train  24.704606  0.883507
Val    24.633193  0.884012
Test   24.653573  0.883602


### Model C Current Price (current_price)

In [22]:
# Features 
cat_features_c = ['charger_type', 'pricing_type', 'network']
num_features_c = ['hour_of_day', 'is_peak_hour', 'power_output_kw', 'ports_total'] # Removed utilization_rate
features_c = num_features_c + cat_features_c

# Data splits
X_train_c, y_train_c = train_df[features_c], train_df['current_price']
X_val_c,   y_val_c   = val_df[features_c],   val_df['current_price']
X_test_c,  y_test_c  = test_df[features_c],  test_df['current_price']

# Pipeline with XGBoost Regressor
preprocessor_c = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features_c),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))]), cat_features_c)
])

model_c = Pipeline([
    ('preprocessor', preprocessor_c),
    ('regressor', XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1, random_state=42))
])

# Train
model_c.fit(X_train_c, y_train_c)

# Apply deterministic rule (Free stations) for evaluation
def apply_free_rule(df, preds):
    return np.where(df['pricing_type'] == 'free', 0.0, np.clip(preds, 0, None))

# Apply the "Free Rule" consistently across Train, Val, and Test (not just Test)
preds_c_train = apply_free_rule(train_df, model_c.predict(X_train_c))
preds_c_val   = apply_free_rule(val_df,   model_c.predict(X_val_c))
preds_c_test  = apply_free_rule(test_df,  model_c.predict(X_test_c))

# Evaluate
print_regression_metrics(
    model_c, 'Model C Final', X_train_c, y_train_c, X_val_c, y_val_c, X_test_c, y_test_c,
    y_pred_train_override=preds_c_train, y_pred_val_override=preds_c_val, y_pred_test_override=preds_c_test
)



--- Model C Final Performance Metrics ---
           RMSE        R2
Train  0.017971  0.987277
Val    0.017979  0.987186
Test   0.017929  0.987358


### Step 5 Save All Models

In [23]:
output_path = '../models/'
os.makedirs(output_path, exist_ok=True)

joblib.dump(model_a, os.path.join(output_path, 'high_utilization_model.pkl'))
joblib.dump(model_b, os.path.join(output_path, 'duration_model.pkl'))
joblib.dump(model_c, os.path.join(output_path, 'price_model.pkl'))

print("\nSUCCESS: All models (A, B, C) have been trained with new algorithms, evaluated, and saved to disk.")


SUCCESS: All models (A, B, C) have been trained with new algorithms, evaluated, and saved to disk.
